## Imports

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from plotnine import *

import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor # Decision Tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor # random forest
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor # gradient boosting
from sklearn.linear_model import LinearRegression

from sklearn import metrics
from sklearn.preprocessing import StandardScaler, LabelBinarizer, OneHotEncoder

from sklearn.model_selection import train_test_split # simple TT split cv
from sklearn.model_selection import KFold # k-fold cv
from sklearn.model_selection import LeaveOneOut #LOO cv

from sklearn.metrics import accuracy_score, confusion_matrix,\
 recall_score, precision_score, roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_transformer
from sklearn.model_selection import GridSearchCV

from sklearn.datasets import make_blobs

%matplotlib inline

## Preprocessing

In [2]:
data = pd.read_csv("https://raw.githubusercontent.com/cmparlettpelleriti/CPSC392ParlettPelleriti/master/Data/streaming.csv")

data.dropna(inplace = True)
data.reset_index(inplace = True)

#Set Up X and Y
predictors = ["gender", "age", "income", "monthssubbed", "plan", "meanhourswatched", "competitorsub", "topgenre", "secondgenre", "numprofiles", "cancelled","downgraded", "bundle", "kids", "longestsession"]
continous = ["age", "income", "monthssubbed", "meanhourswatched", "numprofiles", "longestsession"]

X = data[predictors]
y = data["churn"]

# Validation

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1)

preprocess = make_column_transformer((StandardScaler(), continous),
                            (OneHotEncoder(), ["gender", "plan", "competitorsub","topgenre", "secondgenre", "cancelled", "downgraded", "bundle", "kids"]),
                            remainder = "passthrough")


## Logistic Regression Model

In [3]:
# Create Logitistic Regression Model
lr = LogisticRegression()
pipe = Pipeline([("pre", preprocess), ("model", lr)])

# fit
pipe.fit(X_train, y_train)

# predict
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

y_pred_train_prob = pipe.predict_proba(X_train)[:,1]
y_pred_test_prob = pipe.predict_proba(X_test)[:,1]

# assess
print("Train Acc       : ", accuracy_score(y_train, y_pred_train))
print("Train Precision: ", precision_score(y_train, y_pred_train))
print("Train Recall    : ", recall_score(y_train, y_pred_train))
print("Train ROC AUC   : ", roc_auc_score(y_train, y_pred_train_prob))


print("Test Acc        : ", accuracy_score(y_test, y_pred_test))
print("Test Precision : ", precision_score(y_test, y_pred_test))
print("Test Recall     : ", recall_score(y_test, y_pred_test))
print("Test ROC AUC    : ", roc_auc_score(y_test, y_pred_test_prob))

Train Acc       :  0.7407806721617455
Train Prescision:  0.6047740685281423
Train Recall    :  0.2775374914103238
Train ROC AUC   :  0.7351785632904795
Test Acc        :  0.7437663015127803
Test Prescision :  0.6100079428117554
Test Recall     :  0.2810098792535675
Test ROC AUC    :  0.7416820085487338


## Gradient Boosting Tree Model


In [4]:
# Gradient Boosting Tree
tree = GradientBoostingClassifier()
tree_pipe = Pipeline([("pre", preprocess), ("tree", tree)])

# fit
tree_pipe.fit(X_train, y_train)

# predict
y_pred_train = tree_pipe.predict(X_train)
y_pred_test = tree_pipe.predict(X_test)

y_pred_train_prob = tree_pipe.predict_proba(X_train)[:,1]
y_pred_test_prob = tree_pipe.predict_proba(X_test)[:,1]

# assess
print("Train Acc       : ", accuracy_score(y_train, y_pred_train))
print("Train Prescision: ", precision_score(y_train, y_pred_train))
print("Train Recall    : ", recall_score(y_train, y_pred_train))
print("Train ROC AUC   : ", roc_auc_score(y_train, y_pred_train_prob))


print("Test Acc        : ", accuracy_score(y_test, y_pred_test))
print("Test Prescision : ", precision_score(y_test, y_pred_test))
print("Test Recall     : ", recall_score(y_test, y_pred_test))
print("Test ROC AUC    : ", roc_auc_score(y_test, y_pred_test_prob))

Train Acc       :  0.743238386719067
Train Prescision:  0.6212901413725307
Train Recall    :  0.2682404300901411
Train ROC AUC   :  0.7397644973339611
Test Acc        :  0.7447052686489306
Test Prescision :  0.6211864406779661
Test Recall     :  0.2682034394438346
Test ROC AUC    :  0.7399481302341557


##Calculate the 200 most at risk of Churning from new data set

In [11]:
new_data = pd.read_csv("https://raw.githubusercontent.com/cmparlettpelleriti/CPSC392ParlettPelleriti/master/Data/streamingNEW.csv")
new_data.dropna(inplace = True)
new_data.reset_index(inplace = True)

new_df = new_data[predictors]
new_df["churn"] = pipe.predict_proba(new_df)[:,1]
high_risk = new_df.nlargest(200, "churn")
high_risk.head()

,gender,age,income,monthssubbed,plan,meanhourswatched,competitorsub,topgenre,secondgenre,numprofiles,cancelled,downgraded,bundle,kids,longestsession,churn
42,woman,26.0,57.69,6,B,22.30,0,Comedy,Thriller,3,1,0,0,0,122.39,0.854323
339,woman,20.0,52.05,1,P,9.36,1,Thriller,ScienceFiction,2,1,0,0,1,113.91,0.843520
217,woman,69.0,33.62,4,B,22.90,1,Drama,Comedy,3,0,0,0,0,74.85,0.823021
228,woman,28.0,46.20,4,P,1.65,1,Comedy,Thriller,2,1,1,1,1,142.10,0.783807
103,woman,39.0,52.25,21,B,29.91,1,Thriller,Documentary,1,1,0,0,0,70.09,0.763253


##10 Nearest Neighbors to the 200 at risk


In [17]:
training_data = pd.read_csv("https://raw.githubusercontent.com/cmparlettpelleriti/CPSC392ParlettPelleriti/master/Data/streamingFILMS.csv")


feat = ["age","income","meanhourswatched"]


# Build Empty
z = make_column_transformer((StandardScaler(), feat))
nn = NearestNeighbors(n_neighbors= 10)
pipe = Pipeline([("z", z), ("model", nn)])
pipe.fit(training_data[feat])


# get distances, neighbors
# first grab model to make predictions, then use pre-processor to z score data
distances, neighbors = pipe.named_steps["model"].kneighbors(pipe.named_steps["z"].transform(high_risk[feat]))

# add neighbors to data file
high_risk_data = high_risk.assign(neighbors = list(neighbors))
high_risk_data.head()

# write data to csv
high_risk_data.to_csv("my_recfile.csv", index=False)